# LM-damped CV:  1PA truth  ->  0PA MAP   (all cases)

For each EMRI case, start at the **1PA injected parameters** and iterate the
Levenberg-Marquardt-damped CV / Gauss-Newton step

$$(\Gamma + \lambda\,\mathrm{diag}\,\Gamma)\,\Delta\theta = \langle\partial h_{\rm 0PA}\mid s-h_{\rm 0PA}\rangle,\qquad s=h_{\rm 1PA}(\theta_{\rm tr})$$

up to the 0PA maximum-a-posteriori point.  The iteration uses **only the truth**
and the formalism; the Nelder-Mead `nm_overlap` is carried for the summary
comparison only (it never drives the climb).


In [ ]:
import numpy as np

from few.waveform import GenerateEMRIWaveform
from few.waveform.waveform import SuperKludgeWaveform
from fastlisaresponse import ResponseWrapper
from lisatools.detector import EqualArmlengthOrbits
from lisatools.sensitivity import get_sensitivity, A1TDISens, E1TDISens, T1TDISens
from stableemrifisher.utils import generate_PSD, inner_product, fishinv
from stableemrifisher.fisher import StableEMRIFisher

try:
    import cupy as cp
    xp = cp
except ImportError:
    xp = np
    print("[INFO] CuPy not found, using NumPy instead.")

F_MIN = 1e-5


def _to_float(x):
    return float(x.get()) if hasattr(x, "get") else float(x)


def make_freq_mask(n, dt, fmin):
    return (xp.fft.rfftfreq(n, dt) > fmin)[1:]


def highpass_clip(waveform, dt, fmin):
    n = waveform.shape[-1]
    freq = xp.fft.rfftfreq(n, dt)
    return xp.fft.irfft(xp.fft.rfft(waveform, axis=-1) * (freq >= fmin), n=n, axis=-1)


In [ ]:
use_gpu = True
nchannels = 3
param_names_14 = ["m1", "m2", "a", "p0", "e0", "xI0", "dist", "qS", "phiS",
                  "qK", "phiK", "Phi_phi0", "Phi_theta0", "Phi_r0"]
params_to_infer = ["m1", "m2", "a", "p0", "e0", "qS", "phiS", "Phi_phi0", "Phi_r0"]

CASES = [
    dict(name="idx0", dt=5.0, T=1.0, chi2=0.0, nm_overlap=0.9982389175309955,
         signal_param={"m1": 1000000.0, "m2": 10.0, "a": 0.9, "p0": 7.5, "e0": 0.5,
                       "xI0": 1.0, "dist": 5.0, "qS": 0.7853981633974483, "phiS": 1.0,
                       "qK": 1.0, "phiK": 1.0471975511965976, "Phi_phi0": 0.9,
                       "Phi_theta0": 0.5, "Phi_r0": 0.4, "dev_1": 0.0, "dev_2": 0.0}),
    dict(name="idx9", dt=10.0, T=2.5, chi2=0.95, nm_overlap=0.9980349241125692,
         signal_param={"m1": 1000000.0, "m2": 10.0, "a": 0.9, "p0": 9.07414088, "e0": 0.2,
                       "xI0": 1.0, "dist": 5.0, "qS": 1.04719755, "phiS": 0.785398163,  # dist reduced to raise SNR (ref 13.8096925)
                       "qK": 0.628318531, "phiK": 0.523598776, "Phi_phi0": 0.1,
                       "Phi_theta0": 0.2, "Phi_r0": 0.3, "dev_1": 0.0, "dev_2": 0.0}),
    dict(name="idx13", dt=10.0, T=2.5, chi2=0.95, nm_overlap=0.9997570193374152,
         signal_param={"m1": 1000000.0, "m2": 10.0, "a": 0.5, "p0": 9.97066819, "e0": 0.3,
                       "xI0": 1.0, "dist": 5.0, "qS": 1.04719755, "phiS": 0.785398163,  # dist reduced to raise SNR (ref 9.78949272)
                       "qK": 0.628318531, "phiK": 0.523598776, "Phi_phi0": 0.1,
                       "Phi_theta0": 0.2, "Phi_r0": 0.3, "dev_1": 0.0, "dev_2": 0.0}),
]


In [ ]:
# --- adaptive Levenberg-Marquardt (Nielsen gain-ratio) controls ---
# lambda self-adapts from the gain ratio rho = (actual chi2 drop)/(predicted drop):
#   rho ~ 1 -> model trusted -> lambda shrinks -> bigger, more Gauss-Newton steps
#   rho <= 0 -> step rejected -> lambda grows (x nu, nu doubles) -> smaller, safer steps
# Convergence is on the relative chi2 improvement (fit plateau), NOT the step size.
LAMBDA0, MAX_ITERS, MAX_INNER, REL_TOL = 1e-2, 300, 30, 1e-7
RECOMPUTE_DELTAS_EVERY = 1   # re-optimize SEF finite-diff steps at every point
#                             (=1 is per-point but slow; raise to 5-10 to speed up)


def run_case(case, verbose=True):
    sp, dt, T, chi2 = case["signal_param"], case["dt"], case["T"], case["chi2"]
    t0 = 10000.0
    channels = [A1TDISens, E1TDISens, T1TDISens][:nchannels]
    tdi_chan = {2: "AE", 3: "AET"}[nchannels]
    noise_kwargs = [{"sens_fn": ch} for ch in channels]

    def rkw():
        return dict(Tobs=T, t0=t0, dt=dt, index_lambda=8, index_beta=7, flip_hx=True,
                    is_ecliptic_latitude=False, remove_garbage="zero",
                    orbits=EqualArmlengthOrbits(use_gpu=use_gpu),
                    force_backend="cuda12x" if use_gpu else "cpu",
                    order=20, tdi="1st generation", tdi_chan=tdi_chan)

    wfm = GenerateEMRIWaveform(SuperKludgeWaveform,
                               sum_kwargs=dict(pad_output=True, odd_len=True),
                               return_list=False, use_gpu=use_gpu)
    wresp = ResponseWrapper(waveform_gen=wfm, **rkw())

    def resp_params(inf, evolve_1pa):
        p14 = {n: sp[n] for n in param_names_14}
        p14.update(inf)
        return [p14[n] for n in param_names_14] + [
            chi2, evolve_1pa, False, False, False,
            inf.get("dev_1", sp["dev_1"]), inf.get("dev_2", sp["dev_2"]),
        ]

    def make0(inf):
        return xp.array(wresp(*resp_params(inf, False)))[:nchannels, :]

    true_inf = np.array([sp[n] for n in params_to_infer])
    s = highpass_clip(xp.array(wresp(
        *resp_params(dict(zip(params_to_infer, true_inf)), True)))[:nchannels, :], dt, F_MIN)
    PSD = xp.array(generate_PSD(waveform=s, dt=dt, noise_PSD=get_sensitivity,
                                channels=channels, noise_kwargs=noise_kwargs, use_gpu=use_gpu))
    fmask = make_freq_mask(s.shape[-1], dt, F_MIN)

    def ip(a, b):
        return _to_float(inner_product(a, b, PSD=PSD, dt=dt, freq_mask=fmask, use_gpu=use_gpu))

    def ov(a, b):
        return ip(a, b) / np.sqrt(ip(a, a) * ip(b, b))

    def chi2r(h):
        r = s - h
        return ip(r, r)

    snr = np.sqrt(ip(s, s))

    sef = StableEMRIFisher(
        waveform_class=SuperKludgeWaveform,
        waveform_class_kwargs=dict(sum_kwargs=dict(pad_output=True, odd_len=True)),
        waveform_generator=GenerateEMRIWaveform,
        waveform_generator_kwargs=dict(return_list=False),
        ResponseWrapper=ResponseWrapper, ResponseWrapper_kwargs=rkw(),
        stats_for_nerds=False, use_gpu=use_gpu, deriv_type="stable",
        noise_model=get_sensitivity, noise_kwargs=noise_kwargs, channels=channels,
        T=T, dt=dt, stability_plot=False, der_order=6, Ndelta=12,
        plunge_check=True, return_derivatives=True)
    apa = {"chi2": chi2, "evolve_1PA": False, "evolve_primary": False,
           "evolve_2PA": False, "deviation_included": False, "dev_1": 0.0, "dev_2": 0.0}

    def fish(vec, dl):
        wp = {n: sp[n] for n in param_names_14}
        wp.update(dict(zip(params_to_infer, vec)))
        F = sef(wave_params={n: wp[n] for n in param_names_14}, param_names=params_to_infer,
                add_param_args=apa, deltas=dl, live_dangerously=False, stability_plot=False,
                der_order=8, Ndelta=(20 if dl is None else None))
        return np.asarray(F[-1], dtype=float), xp.array(F[0])

    npar = len(params_to_infer)
    cur = true_inf.astype(float).copy()          # START AT THE TRUTH
    dl = None
    lam, nu = LAMBDA0, 2.0
    ov_truth = ov(s, make0(dict(zip(params_to_infer, cur))))
    converged, nit = False, 0
    for it in range(MAX_ITERS):
        nit = it + 1
        if it % RECOMPUTE_DELTAS_EVERY == 0:
            dl = None                     # re-optimize finite-diff steps at this new point
        G, dH = fish(cur, dl)
        if dl is None:
            dl = sef.deltas
        h = make0(dict(zip(params_to_infer, cur)))
        r = s - h
        g = np.array([ip(dH[j], r) for j in range(npar)])           # <d_j h | s - h>
        sig = np.sqrt(np.abs(np.diag(fishinv(cur[0], G, index_of_M=0))))
        c0 = chi2r(h)
        dvec = np.abs(np.diag(G)) + 1e-30                            # Marquardt scaling (>0)

        # adaptive inner loop: gain ratio rho self-tunes the step size (lambda, nu)
        delta, ok, rel = np.zeros(npar), False, 0.0
        for _ in range(MAX_INNER):
            try:
                delta = np.linalg.solve(G + lam * np.diag(dvec), g)
            except np.linalg.LinAlgError:
                lam *= nu; nu *= 2.0; continue
            c1 = chi2r(make0(dict(zip(params_to_infer, cur + delta))))
            predicted = float(delta @ (g + lam * dvec * delta))     # LM-model chi2 reduction
            rho = (c0 - c1) / predicted if predicted > 0 else -1.0
            if rho > 0.0:                                           # accept: shrink lambda
                lam *= max(1.0 / 3.0, 1.0 - (2.0 * rho - 1.0) ** 3)
                nu = 2.0
                rel = (c0 - c1) / c0
                ok = True
                break
            lam *= nu; nu *= 2.0                                    # reject: grow lambda, smaller step
        step = float(np.max(np.abs(delta / sig)))
        if verbose:
            print(f"    it {it:>3} lam={lam:.2e} ov={ov(s, h):.6f} chi2={c0:.4e} "
                  f"|d/sig|={step:.2e} rel_dchi2={rel:.1e}")
        if not ok:
            break
        cur = cur + delta
        if rel < REL_TOL:                                          # converged: chi2 plateaued
            converged = True
            break

    hf = make0(dict(zip(params_to_infer, cur)))
    return dict(name=case["name"], snr=float(snr), ov_truth=float(ov_truth),
                ov_final=float(ov(s, hf)), chi2_final=float(chi2r(hf)),
                converged=converged, nit=nit, params=cur.copy(), true=true_inf,
                nm_overlap=case.get("nm_overlap"))


In [ ]:
results = []
for case in CASES:
    print(f"=== {case['name']} (SNR_ref via dist={case['signal_param']['dist']}) ===")
    res = run_case(case, verbose=True)
    results.append(res)
    print(f"  -> ov(truth)={res['ov_truth']:.4f}  ov(MAP)={res['ov_final']:.6f}  "
          f"converged={res['converged']} ({res['nit']} it)\n")


In [ ]:
print(f"{'case':6} {'SNR':>7} {'ov@truth':>10} {'ov@CV-MAP':>12} "
      f"{'ov@NM':>10} {'conv':>6} {'verdict':>9}")
for r in results:
    nm = r["nm_overlap"]
    verdict = ("SUCCESS" if (r["converged"] and r["ov_final"] > 0.99)
               else ("STALL" if r["ov_final"] < 0.9 else "PARTIAL"))
    tag = "" if nm is None else (" (CV>=NM)" if r["ov_final"] >= nm - 1e-6 else " (CV<NM)")
    nm_s = f"{nm:.6f}" if nm is not None else "-"
    print(f"{r['name']:6} {r['snr']:>7.2f} {r['ov_truth']:>10.4f} {r['ov_final']:>12.6f} "
          f"{nm_s:>10} {str(r['converged']):>6} {verdict:>9}{tag}")

print("\nsystematic bias  (CV-MAP - truth):")
for r in results:
    print(f"[{r['name']}]  chi2_final=<r|r>={r['chi2_final']:.4e}")
    for nm_, tv, cv in zip(params_to_infer, r["true"], r["params"]):
        print(f"    {nm_:10s} true={tv: .8e}  MAP={cv: .8e}  bias={cv - tv: .3e}")
